#  Transformer Fine-Tuning (RoBERTa)

Train a transformer-based model for sentiment classification using rating-based labels,
and compare its performance with traditional ML models.

In [49]:
#Import Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

In [50]:
# Load data

df = pd.read_csv('/Users/olgabencomo/Desktop/Proyectos Portafolio/Sentiment Analysis Project/data/processed/clean_data.csv')

In [51]:
df = df.sample(4000, random_state=42)

In [52]:
df

,Review Text,Rating,Clean_Review,sentiment
12888,This sweater is so beautiful on. it is thick m...,5,sweater beautiful thick material make look box...,Positive
19134,This piece is almost what i want... i tried on...,4,piece almost want tried white version xs felt ...,Positive
18051,Really like this blouse but am returning for a...,4,really like blouse returning larger size much ...,Positive
10263,These are the perfect light weight relaxing su...,5,perfect light weight relaxing summer pants fab...,Positive
7097,These look nothing like the picture! they are ...,2,look nothing like picture super highwaisted lo...,Negative
...,...,...,...,...
12451,Dress is true to size. has a built in liner wi...,4,dress true size built liner thin straps see re...,Positive
14338,This is a truly exquisite top. it looks exactl...,5,truly exquisite top looks exactly online casca...,Positive
2497,Love this dress. i have it in white. just wash...,5,love dress white washed cold water machine sti...,Positive
15324,These capri length pants are awesome! i do re...,5,capri length pants awesome recommend sizing th...,Positive


In [53]:
label_map = {
    'Negative': 0,
    'Neutral': 1,
    'Positive': 2
}

df['label'] = df['sentiment'].map(label_map)

In [54]:
df = df[['Clean_Review', 'label']]

In [55]:
#Split the data

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

In [56]:
train_dataset = Dataset.from_pandas(train_df[['Clean_Review','label']])
test_dataset = Dataset.from_pandas(test_df[['Clean_Review','label']])

In [57]:
#Tokenization

tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

def tokenize(example):
    return tokenizer(
        example['Clean_Review'],
        truncation=True,
        padding='max_length',
        max_length=64   
    )



In [58]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 800/800 [00:00<00:00, 23759.39 examples/s]


In [59]:
#Model

model = AutoModelForSequenceClassification.from_pretrained(
    "distilroberta-base",
    num_labels=3
)

Loading weights: 100%|██████████| 101/101 [00:00<00:00, 3030.67it/s]
RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [60]:
#Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,   
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none"
)

In [61]:
#Define metrics


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    from sklearn.metrics import f1_score

    return {
        "f1_macro": f1_score(labels, preds, average='macro')
    }

In [62]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [63]:
trainer.train()

/Users/olgabencomo/Desktop/Proyectos Portafolio/Sentiment Analysis Project/sentiment_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1 Macro
1,0.498934,0.514411,0.518192


TrainOutput(global_step=400, training_loss=0.6000836181640625, metrics={'train_runtime': 334.5953, 'train_samples_per_second': 9.564, 'train_steps_per_second': 1.195, 'total_flos': 52987904409600.0, 'train_loss': 0.6000836181640625, 'epoch': 1.0})

In [64]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred))
print("Accuracy:", accuracy_score(y_true, y_pred))

/Users/olgabencomo/Desktop/Proyectos Portafolio/Sentiment Analysis Project/sentiment_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.41      0.38      0.39        82
           1       0.36      0.19      0.25       104
           2       0.87      0.95      0.91       614

    accuracy                           0.79       800
   macro avg       0.55      0.51      0.52       800
weighted avg       0.76      0.79      0.77       800

Accuracy: 0.7925


### Computational Constraints

Due to limited computational resources, the dataset was reduced to enable feasible training of the transformer model.

While this approach significantly decreased training time, it also limited the model's ability to fully learn complex patterns from the data.

This trade-off highlights the importance of balancing computational efficiency with model performance in real-world scenarios.

## Transformer vs Classical Model Comparison

The transformer model (DistilRoBERTa) achieved an accuracy of 0.79 and a macro F1-score of 0.52.

### Key Findings:
- Strong performance on the Positive class (F1: 0.91)
- Lower performance on Negative and Neutral classes
- Overall performance was slightly lower than the SVM baseline

### Insights:
- Transformer models require larger datasets to outperform classical approaches
- Due to computational constraints, the dataset was reduced to make training feasible
- This reduction likely limited the model’s ability to learn complex patterns
- Class imbalance and label noise also impacted performance

### Final Takeaway:
In this project, the SVM model outperformed the transformer model, demonstrating that simpler models can be more effective when working with limited data and computational resources.

Future improvements could include:
- Training on the full dataset
- Increasing computational resources (GPU)
- Improving label quality
- Performing deeper fine-tuning of transformer models